## 1. Install Libraries

In [ ]:
# Install PyTorch 2.5.0 + CUDA 12.4
!pip install -q torch==2.5.0+cu124 torchvision==0.20.0+cu124 --extra-index-url https://download.pytorch.org/whl/cu124

# Install PyG dependencies
!pip install -q torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.5.0+cu124.html

# Install PyG
!pip install -q torch-geometric

# Install other libraries
!pip install -q scikit-learn matplotlib pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 111.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Import Libraries

In [ ]:
import torch
import warnings
warnings.filterwarnings('ignore')

## 3. Upload Python Files
Upload: models.py, dataset.py, metrics.py, test_model.py

In [ ]:
from google.colab import files
print("Upload Python files (models.py, dataset.py, metrics.py, test_model.py):")
uploaded = files.upload()

Upload Python files (models.py, dataset.py, metrics.py, test_model.py):


Saving dataset.py to dataset.py
Saving metrics.py to metrics.py
Saving models.py to models.py
Saving test_model.py to test_model.py


## 4. Upload ENZYMES Dataset Files

In [ ]:
print("Upload ENZYMES dataset files (5 txt files):")
uploaded_data = files.upload()

Upload ENZYMES dataset files (5 txt files):


Saving ENZYMES_A.txt to ENZYMES_A.txt
Saving ENZYMES_graph_indicator.txt to ENZYMES_graph_indicator.txt
Saving ENZYMES_graph_labels.txt to ENZYMES_graph_labels.txt
Saving ENZYMES_node_attributes.txt to ENZYMES_node_attributes.txt
Saving ENZYMES_node_labels.txt to ENZYMES_node_labels.txt


## 5. Create ENZYMES Directory

In [ ]:
!mkdir -p ENZYMES
!mv ENZYMES_*.txt ENZYMES/

## 6. Upload Saved Models
Upload: gcn1_best.pth, gcn2_best.pth, gcn3_best.pth

In [ ]:
print("Upload saved model files (.pth):")
uploaded_models = files.upload()

Upload saved model files (.pth):


Saving gcn1_best.pth to gcn1_best.pth
Saving gcn2_best.pth to gcn2_best.pth
Saving gcn3_best.pth to gcn3_best.pth


## 7. Import Custom Modules

In [ ]:
from models import GraphClassifierGCN1, GraphClassifierGCN2, GraphClassifierGCN3
from dataset import load_enzymes_raw, split_dataset, get_data_loaders, get_dataset_info
from metrics import compute_metrics
from test_model import test_model

## 8. Setup Device

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 9. Load Dataset and Create Test Loader

In [ ]:
# Load dataset
dataset = load_enzymes_raw('ENZYMES')
print(f'Total graphs: {len(dataset)}')

# Get dataset info
info = get_dataset_info(dataset)
input_dim = info['num_features']
num_classes = info['num_classes']

print(f'Input dim: {input_dim}, Num classes: {num_classes}')

Total graphs: 600
Input dim: 19, Num classes: 6


## 10. Split Dataset (Same as Training)

In [ ]:
# Split with same seed as training
train_dataset, val_dataset, test_dataset = split_dataset(dataset, train_ratio=0.8, val_ratio=0.1, seed=42)
print(f'Test set size: {len(test_dataset)}')

Test set size: 60


## 11. Create Test DataLoader

In [ ]:
_, _, test_loader = get_data_loaders(train_dataset, val_dataset, test_dataset, batch_size=32)
print(f'Test loader created with {len(test_loader)} batches')

Test loader created with 2 batches


## 12. Define Hyperparameters

In [ ]:
hidden_dim = 128
dropout_rate = 0.5

print(f'Model config: hidden_dim={hidden_dim}, dropout={dropout_rate}')

Model config: hidden_dim=128, dropout=0.5


## 13. Test GCN1

In [ ]:
print("Testing GCN1...")
test_results_gcn1 = test_model(
    'gcn1_best.pth', test_loader, GraphClassifierGCN1,
    input_dim, hidden_dim, num_classes, device, dropout_rate
)

metrics_gcn1 = compute_metrics(test_results_gcn1['labels'], test_results_gcn1['logits'])
print(f"GCN1 - Accuracy: {metrics_gcn1['accuracy']:.4f}, F1: {metrics_gcn1['f1_score']:.4f}, AUC: {metrics_gcn1['auc']:.4f}")

Testing GCN1...
GCN1 - Accuracy: 0.2500, F1: 0.2855, AUC: 0.6525


## 14. Test GCN2

In [ ]:
print("Testing GCN2...")
test_results_gcn2 = test_model(
    'gcn2_best.pth', test_loader, GraphClassifierGCN2,
    input_dim, hidden_dim, num_classes, device, dropout_rate
)

metrics_gcn2 = compute_metrics(test_results_gcn2['labels'], test_results_gcn2['logits'])
print(f"GCN2 - Accuracy: {metrics_gcn2['accuracy']:.4f}, F1: {metrics_gcn2['f1_score']:.4f}, AUC: {metrics_gcn2['auc']:.4f}")

Testing GCN2...
GCN2 - Accuracy: 0.3833, F1: 0.3812, AUC: 0.7753


## 15. Test GCN3

In [ ]:
print("Testing GCN3...")
test_results_gcn3 = test_model(
    'gcn3_best.pth', test_loader, GraphClassifierGCN3,
    input_dim, hidden_dim, num_classes, device, dropout_rate
)

metrics_gcn3 = compute_metrics(test_results_gcn3['labels'], test_results_gcn3['logits'])
print(f"GCN3 - Accuracy: {metrics_gcn3['accuracy']:.4f}, F1: {metrics_gcn3['f1_score']:.4f}, AUC: {metrics_gcn3['auc']:.4f}")

Testing GCN3...
GCN3 - Accuracy: 0.4667, F1: 0.4827, AUC: 0.8010


## 16. Final Comparison Table

In [ ]:
import pandas as pd

results = pd.DataFrame({
    'Model': ['GCN-1 layer', 'GCN-2 layer', 'GCN-3 layer'],
    'F1': [metrics_gcn1['f1_score'], metrics_gcn2['f1_score'], metrics_gcn3['f1_score']],
    'Accuracy': [metrics_gcn1['accuracy'], metrics_gcn2['accuracy'], metrics_gcn3['accuracy']],
    'AUC': [metrics_gcn1['auc'], metrics_gcn2['auc'], metrics_gcn3['auc']]
})

print("FINAL TEST SET RESULTS")
print("."*60)
print(results.to_string(index=False))
print("."*60)

FINAL TEST SET RESULTS
............................................................
      Model       F1  Accuracy      AUC
GCN-1 layer 0.285478  0.250000 0.652469
GCN-2 layer 0.381220  0.383333 0.775263
GCN-3 layer 0.482692  0.466667 0.800981
............................................................
